# LLM Uncertainty Benchmark — Kaggle GPU

Runs the full CP benchmark with **open-source models on Kaggle's free GPU** (T4 x2 or P100).  
Uses HuggingFace Transformers for **direct logit extraction** — more accurate than continuation scoring.

**Models:** Qwen3-0.6B, Qwen2.5-3B, Gemma-3-1B, Llama-3.2-1B  
*(all fit in 16GB GPU RAM; larger models need GPU T4 x2)*

**Setup:**
1. New notebook → Settings → Accelerator → **GPU T4 x2**
2. Add secret: `HF_TOKEN` (from huggingface.co → Settings → Access Tokens)
3. Run all cells

**Time estimate:** ~1–2 hours for 4 models × 5 tasks × n=100

## 1. Install dependencies

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'accelerate', 'bitsandbytes',
                'numpy', 'scikit-learn', 'tqdm', 'matplotlib'], check=True)
print('Done.')

## 2. Clone repo

In [ ]:
import os, subprocess

if not os.path.exists('LLM-Uncertainty-Study'):
    subprocess.run(['git', 'clone', 'https://github.com/SokhengDin/LLM-Uncertainty-Study.git'], check=True)
os.chdir('LLM-Uncertainty-Study')
print('Working directory:', os.getcwd())

## 3. Config — models and tasks

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

# HuggingFace token (needed for gated models like Llama)
try:
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    os.environ['HF_TOKEN'] = HF_TOKEN
    print('HF_TOKEN loaded from Kaggle secrets')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
    print('HF_TOKEN not found in secrets — gated models may fail')

# ── Models — all fit on T4 16GB ──────────────────────────────────────────────
# Format: (hf_model_id, short_name)
# All models below are fully PUBLIC (no HF gating, no approval needed):
#   - Qwen3-0.6B, Qwen2.5-3B  → Apache 2.0, ungated
#   - gemma-2b-it              → Google license, ungated (gemma-3-* are gated)
#   - OLMo-1B-Instruct         → Apache 2.0, fully open, replaces gated Llama
MODELS = [
    ('Qwen/Qwen3-0.6B',                    'qwen3-0.6b'),
    ('Qwen/Qwen2.5-3B-Instruct',           'qwen2.5-3b'),
    ('google/gemma-2b-it',                 'gemma-2b'),
    ('allenai/OLMo-1B-hf',                 'olmo-1b'),
    # Uncomment for T4 x2 (32GB total) — all ungated:
    # ('Qwen/Qwen2.5-7B-Instruct',         'qwen2.5-7b'),
    # ('google/gemma-7b-it',               'gemma-7b'),
    # ('allenai/OLMo-7B-Instruct-hf',      'olmo-7b'),
    # ── Gated (need HF approval + HF_TOKEN secret) ──
    # ('meta-llama/Llama-3.2-1B-Instruct', 'llama3.2-1b'),
    # ('meta-llama/Llama-3.1-8B-Instruct', 'llama3.1-8b'),
    # ('google/gemma-3-1b-it',             'gemma-3-1b'),
]

DATASETS = [
    'mmlu_10k',
    'cosmosqa_10k',
    'hellaswag_10k',
    'halu_dialogue',
    'halu_summarization',
]

SAMPLES  = 100    # n_cal ≈ 52 → qhat ≈ 0.923 (non-trivial CP)
ALPHA    = 0.1
PROMPT   = 'base'
ICL      = 'icl1'
OUT_DIR  = 'outputs_kaggle'
FIG_DIR  = 'figures_kaggle'

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
print(f'Config: {len(MODELS)} models × {len(DATASETS)} tasks × n={SAMPLES}')
print('Models:', [m[1] for m in MODELS])

## 4. HuggingFace logit extractor
Direct last-token logits — more accurate than continuation scoring.

In [ ]:
import json, pickle, random, sys
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.notebook import tqdm

CHOICES = ['A', 'B', 'C', 'D', 'E', 'F']

FEW_SHOT_IDS = {
    'MMLU'               : [1, 3, 5, 7, 9],
    'HellaSwag'          : [1, 3, 5, 7, 9],
    'CosmosQA'           : [1, 3, 5, 7, 9],
    'Halu-OpenDialKG'    : [5, 7, 9],
    'Halu-CNN/DailyMail' : [9],
}
FEW_SHOT_RESERVE = 10
IDS_TO_REMOVE    = [1, 3, 5, 7, 9]  # same as main.py


def load_data(path, max_samples):
    data = json.load(open(path))
    few  = data[:FEW_SHOT_RESERVE]
    rest = data[FEW_SHOT_RESERVE:]
    if len(rest) > max_samples:
        random.seed(42)
        rest = random.sample(rest, max_samples)
    print(f'  {FEW_SHOT_RESERVE} few-shot + {len(rest)} test = {len(few+rest)} total')
    return few + rest


def fmt_example(ex, prompt, with_answer=False):
    src = ex['source']
    if src == 'MMLU':
        prompt += 'Question: ' + ex['question'] + '\nChoices:\n'
    elif src in ('CosmosQA', 'HellaSwag'):
        prompt += 'Context: ' + ex['context'] + '\n'
        prompt += 'Question: ' + ex['question'] + '\nChoices:\n'
    elif src == 'Halu-OpenDialKG':
        prompt += 'Dialogue: ' + ex['context'] + '\n'
        prompt += 'Question: ' + ex['question'] + '\nChoices:\n'
    elif src == 'Halu-CNN/DailyMail':
        prompt += 'Document: ' + ex['context'] + '\n'
        prompt += 'Question: ' + ex['question'] + '\nChoices:\n'
    for k, v in ex['choices'].items():
        prompt += f'{k}. {v}\n'
    prompt += 'Answer:'
    if with_answer:
        prompt += ' ' + ex['answer'] + '\n'
    return prompt


def build_prompts(data):
    src   = data[0]['source']
    fsids = FEW_SHOT_IDS[src]
    shots = [data[i] for i in fsids]
    out   = []
    for ex in data:
        if ex['id'] in IDS_TO_REMOVE:
            continue
        p = ''
        for fs in shots:
            p = fmt_example(fs, p, with_answer=True)
        out.append({'id': ex['id'], 'prompt': fmt_example(ex, p)})
    return out


class HFLogitExtractor:
    """Extract last-token logprobs for A-F using HuggingFace model."""

    def __init__(self, model_id, hf_token=''):
        print(f'Loading {model_id}...')
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id, token=hf_token or None, trust_remote_code=True)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id, token=hf_token or None,
            torch_dtype=torch.float16,
            device_map='auto',
            trust_remote_code=True,
        )
        self.model.eval()

        # Get token IDs for A-F (take last token of each option string)
        self.choice_ids = [
            self.tokenizer.encode(f' {c}', add_special_tokens=False)[-1]
            for c in CHOICES
        ]
        print(f'  Choice token IDs: {dict(zip(CHOICES, self.choice_ids))}')

    def get_choice_logits(self, prompt: str) -> np.ndarray:
        inputs = self.tokenizer(
            prompt, return_tensors='pt', truncation=True, max_length=2048
        ).to(self.model.device)
        with torch.no_grad():
            out = self.model(**inputs)
        # Last token logits over full vocab
        last_logits = out.logits[0, -1, :].float().cpu()
        return last_logits[self.choice_ids].numpy()

    def unload(self):
        import gc
        del self.model
        gc.collect()
        torch.cuda.empty_cache()
        print('  Model unloaded from GPU.')


print('Logit extractor ready.')

## 5. Generate logits — all models × all tasks

In [ ]:
for hf_id, short in MODELS:
    # Check if all datasets already done
    all_done = all(
        os.path.exists(f'{OUT_DIR}/{short}_{ds}_base_icl1_sample{SAMPLES}.pkl')
        for ds in DATASETS
    )
    if all_done:
        print(f'SKIP {short} (all datasets done)')
        continue

    extractor = HFLogitExtractor(hf_id, HF_TOKEN)

    for ds in DATASETS:
        pkl = f'{OUT_DIR}/{short}_{ds}_base_icl1_sample{SAMPLES}.pkl'
        if os.path.exists(pkl):
            print(f'  SKIP {short} | {ds}')
            continue

        print(f'\n--- {short} | {ds} ---')
        data    = load_data(f'data/{ds}.json', SAMPLES)
        prompts = build_prompts(data)

        outputs = []
        for ex in tqdm(prompts, desc=f'{short}|{ds}'):
            logits = extractor.get_choice_logits(ex['prompt'])
            outputs.append({'id': ex['id'], 'logits_options': logits})

        with open(pkl, 'wb') as f:
            pickle.dump(outputs, f)
        print(f'  Saved → {pkl}')

    extractor.unload()

print('\nAll logits generated!')

## 6. CP evaluation

In [ ]:
import subprocess, sys

for hf_id, short in MODELS:
    print(f'\n=== Evaluating {short} ===')
    subprocess.run([
        sys.executable, 'main.py',
        '--model',           short,
        '--data_names',      *DATASETS,
        '--prompt_methods',  PROMPT,
        '--icl_methods',     ICL,
        '--max_samples',     str(SAMPLES),
        '--alpha',           str(ALPHA),
        '--logits_data_dir', OUT_DIR,
        '--output_dir',      OUT_DIR,
    ], check=False)

## 7. Generate figures

In [ ]:
subprocess.run([
    sys.executable, 'plot_results.py',
    '--samples',     str(SAMPLES),
    '--prompt',      PROMPT,
    '--icl',         ICL,
    '--results_dir', OUT_DIR,
    '--figures_dir', FIG_DIR,
], check=False)

## 8. Display figures

In [ ]:
from IPython.display import Image, display
import glob

for png in sorted(glob.glob(f'{FIG_DIR}/*.png')):
    print(f'\n{png}')
    display(Image(png))

## 9. Summary table

In [ ]:
import json, numpy as np

key = f'{PROMPT}_{ICL}'
col = 16
print('RESULTS  (CR% / Acc% / SS)')
print(f"{'Model':<20}" + ''.join(f"{d.split('_')[0]:>{col}}" for d in DATASETS))
print('-' * (20 + col * len(DATASETS)))

for hf_id, short in MODELS:
    path = f'{OUT_DIR}/{short}_all_results.json'
    if not os.path.exists(path):
        print(f'{short:<20}  (no results)'); continue
    res = json.load(open(path))
    row = f'{short:<20}'
    for d in DATASETS:
        if d not in res or key not in res[d].get('Acc', {}):
            row += f"{'N/A':>{col}}"; continue
        acc = 100 * res[d]['Acc'][key]
        cr  = 100 * np.mean([res[d]['LAC_coverage'][key], res[d]['APS_coverage'][key]])
        ss  =       np.mean([res[d]['LAC_set_size'][key],  res[d]['APS_set_size'][key]])
        row += f"{cr:.0f}/{acc:.0f}/{ss:.1f}".rjust(col)
    print(row)

print(f'\nn={SAMPLES} → n_cal≈52 → α={ALPHA}')

## 10. Download results

In [ ]:
import shutil

shutil.make_archive('kaggle_results', 'zip', '.', OUT_DIR)
shutil.make_archive('figures_kaggle', 'zip', '.', FIG_DIR)
print('Outputs zipped → kaggle_results.zip')
print('Figures  zipped → figures_kaggle.zip')
print('Download from: Notebook → Output tab → <filename>.zip')

## 11. Generate logits — shared & task prompt variants (for Weighted CP)

Run this section **after** Section 5 completes.  
We generate logits for the two additional prompt prefixes used in Ye et al.:
- `shared_icl1` — shared instruction + few-shot
- `task_icl1`   — task-specific instruction + few-shot

Same questions, same models, same n=100.  
These paired logits let us compute the density ratio $w(x) = dP_{k_\text{test}}/dP_{k_\text{cal}}(x)$.

In [ ]:
import json, pickle, random, os, sys
import numpy as np
import torch
from tqdm.notebook import tqdm

# ── Prompt prefixes (from utils/prompt.py) ────────────────────────────────────
SHARED_ICL1_PREFIX = (
    "Below are some examples of multiple-choice questions with six potential answers. "
    "For each question, only one option is correct.\n\n"
)

TASK_ICL1_PREFIX = {
    "MMLU":
        "Below are some examples of multiple-choice questions about question answering. "
        "Each question should be answered based on your world knowledge and problem solving ability.\n\n",
    "HellaSwag":
        "Below are some examples of multiple-choice questions about commonsense natural language inference. "
        "For each question, there is a given context and the answer is the option that most likely follows the context.\n\n",
    "CosmosQA":
        "Below are some examples of multiple-choice questions about reading comprehension. "
        "Each question should be answered based on the given context and commonsense reasoning when necessary.\n\n",
    "Halu-OpenDialKG":
        "Below are some examples of multiple-choice questions about dialogue response selection. "
        "For each question, the answer is the option that represents the most suitable response "
        "for the given dialogue history, without hallucination and non-factual information.\n\n",
    "Halu-CNN/DailyMail":
        "Below are some examples of multiple-choice questions about document summarization. "
        "For each question, the answer is the option that accurately summarizes the given document "
        "without hallucination and non-factual information.\n\n",
}

# Source name → TASK_ICL1_PREFIX key (matches data['source'])
DS_TO_SRC = {
    'mmlu_10k':           'MMLU',
    'hellaswag_10k':      'HellaSwag',
    'cosmosqa_10k':       'CosmosQA',
    'halu_dialogue':      'Halu-OpenDialKG',
    'halu_summarization': 'Halu-CNN/DailyMail',
}


def build_prompts_with_prefix(data, prefix):
    """Build few-shot prompts with a custom instruction prefix."""
    src    = data[0]['source']
    fsids  = FEW_SHOT_IDS[src]
    shots  = [data[i] for i in fsids]
    out    = []
    for ex in data:
        if ex['id'] in IDS_TO_REMOVE:
            continue
        p = prefix          # <-- only difference from build_prompts()
        for fs in shots:
            p = fmt_example(fs, p, with_answer=True)
        out.append({'id': ex['id'], 'prompt': fmt_example(ex, p)})
    return out


EXTRA_PROMPTS = [
    ('shared', 'icl1'),   # shared instruction + few-shot
    ('task',   'icl1'),   # task-specific instruction + few-shot
]

print('Generating logits for shared_icl1 and task_icl1 prompt variants...')
print(f'Models: {[m[1] for m in MODELS]}')
print(f'Datasets: {DATASETS}')
print()

for hf_id, short in MODELS:
    # Check if all extra variants already exist
    all_done = all(
        os.path.exists(f'{OUT_DIR}/{short}_{ds}_{p}_{icl}_sample{SAMPLES}.pkl')
        for ds in DATASETS
        for p, icl in EXTRA_PROMPTS
    )
    if all_done:
        print(f'SKIP {short} (all extra variants done)')
        continue

    extractor = HFLogitExtractor(hf_id, HF_TOKEN)

    for ds in DATASETS:
        data = load_data(f'data/{ds}.json', SAMPLES)
        src  = data[0]['source']

        for prompt_type, icl in EXTRA_PROMPTS:
            pkl = f'{OUT_DIR}/{short}_{ds}_{prompt_type}_{icl}_sample{SAMPLES}.pkl'
            if os.path.exists(pkl):
                print(f'  SKIP {short} | {ds} | {prompt_type}_{icl}')
                continue

            # Select prefix
            if prompt_type == 'shared':
                prefix = SHARED_ICL1_PREFIX
            else:
                prefix = TASK_ICL1_PREFIX[DS_TO_SRC[ds]]

            print(f'\n--- {short} | {ds} | {prompt_type}_{icl} ---')
            prompts = build_prompts_with_prefix(data, prefix)

            outputs = []
            for ex in tqdm(prompts, desc=f'{short}|{ds}|{prompt_type}'):
                logits = extractor.get_choice_logits(ex['prompt'])
                outputs.append({'id': ex['id'], 'logits_options': logits})

            with open(pkl, 'wb') as f:
                pickle.dump(outputs, f)
            print(f'  Saved → {pkl}')

    extractor.unload()

print('\nAll extra prompt logits generated!')

## 12. Weighted CP — Standard vs Weighted across all cross-prompt pairs

For each (model, dataset), we now have **3 prompt variants** → **6 cross-prompt pairs**:
- base → shared, base → task, shared → task (and their reverses)

For each pair: calibrate on prompt A, test on prompt B.
- **Standard CP**: ignores the prompt shift → coverage may drop below 90%
- **Weighted CP**: estimates $\hat{w}(x)$ via logistic regression on softmax features → restores coverage

This is the core empirical validation of Proposition 5.1 in the report.

In [ ]:
import json, pickle, random, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from itertools import permutations

CHOICES  = ['A', 'B', 'C', 'D', 'E', 'F']
ALPHA    = 0.1
SEED     = 42

# ── helpers ───────────────────────────────────────────────────────────────────
def softmax(x):
    e = np.exp(np.array(x, dtype=float) - np.max(x))
    return e / e.sum()

def lac_score(probs, answer):
    return 1.0 - probs[CHOICES.index(answer)]

def load_paired(short, ds, prompt_type, out_dir, samples=100, seed=42):
    """Load logits pkl + data json, return list of (data_item, logits_row)."""
    pkl  = f'{out_dir}/{short}_{ds}_{prompt_type}_icl1_sample{samples}.pkl'
    data = json.load(open(f'data/{ds}.json'))
    if not os.path.exists(pkl):
        return None

    logits_all = pickle.load(open(pkl, 'rb'))
    rest = data[10:]
    random.seed(seed)
    if len(rest) > samples:
        rest = random.sample(rest, samples)
    demo_ids = {1, 3, 5, 7, 9}
    rest = [d for d in rest if d['id'] not in demo_ids]

    logits_by_id = {str(r['id']): r for r in logits_all}
    paired = [(d, logits_by_id[str(d['id'])]) for d in rest if str(d['id']) in logits_by_id]
    random.seed(seed)
    random.shuffle(paired)
    return paired

def estimate_weights(cal_feat, test_feat):
    X = np.vstack([cal_feat, test_feat])
    y = np.array([0]*len(cal_feat) + [1]*len(test_feat))
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    clf = LogisticRegression(max_iter=500, C=1.0)
    clf.fit(Xs, y)
    p1 = clf.predict_proba(Xs[:len(cal_feat)])[:, 1]
    p0 = 1.0 - p1 + 1e-9
    return np.clip((p1 / p0) * (len(cal_feat) / len(test_feat)), 1e-3, 1e3)

def weighted_quantile(scores, weights, alpha):
    scores  = np.array(scores, dtype=float)
    weights = np.array(weights, dtype=float)
    weights = weights / weights.sum()
    aug_s = np.append(scores, np.inf)
    aug_w = np.append(weights, 1.0 / len(weights))
    aug_w = aug_w / aug_w.sum()
    order = np.argsort(aug_s)
    cumw  = np.cumsum(aug_w[order])
    idx   = np.searchsorted(cumw, 1.0 - alpha)
    return float(aug_s[order][min(idx, len(aug_s)-1)])

def run_cp_pair(cal_pairs, test_pairs, alpha=0.1):
    """Returns standard and weighted CP results for one (cal_prompt, test_prompt) pair."""
    cal_scores = [lac_score(softmax(r['logits_options']), d['answer']) for d, r in cal_pairs]
    n     = len(cal_scores)
    q_lvl = np.ceil((n + 1) * (1 - alpha)) / n
    qhat  = np.quantile(cal_scores, min(q_lvl, 1.0), method='higher')

    cal_feat  = np.array([softmax(r['logits_options']) for _, r in cal_pairs])
    test_feat = np.array([softmax(r['logits_options']) for _, r in test_pairs])
    weights   = estimate_weights(cal_feat, test_feat)
    qhat_w    = weighted_quantile(cal_scores, weights, alpha)

    std_cov, wcp_cov = [], []
    std_ss,  wcp_ss  = [], []
    for item, row in test_pairs:
        probs = softmax(row['logits_options'])
        ps_std = [CHOICES[i] for i, p in enumerate(probs) if p >= 1 - qhat]  or [CHOICES[np.argmax(probs)]]
        ps_wcp = [CHOICES[i] for i, p in enumerate(probs) if p >= 1 - qhat_w] or [CHOICES[np.argmax(probs)]]
        std_cov.append(int(item['answer'] in ps_std)); std_ss.append(len(ps_std))
        wcp_cov.append(int(item['answer'] in ps_wcp)); wcp_ss.append(len(ps_wcp))

    return {
        'std':  {'cr': np.mean(std_cov)*100, 'ss': np.mean(std_ss), 'qhat': qhat},
        'wcp':  {'cr': np.mean(wcp_cov)*100, 'ss': np.mean(wcp_ss), 'qhat_w': qhat_w},
    }

# ── Run all cross-prompt pairs ────────────────────────────────────────────────
PROMPT_TYPES = ['base', 'shared', 'task']
DS_LABELS = {
    'mmlu_10k': 'MMLU', 'hellaswag_10k': 'HellaSwag',
    'cosmosqa_10k': 'CosmosQA', 'halu_dialogue': 'HaluDial',
    'halu_summarization': 'HaluSum',
}
MODEL_LABELS = {
    'qwen3-0.6b': 'Qwen3-0.6B', 'qwen2.5-3b': 'Qwen2.5-3B',
    'qwen2.5-7b': 'Qwen2.5-7B', 'olmo-1b': 'OLMo-1B', 'olmo-7b': 'OLMo-7B',
}

rows = []
for hf_id, short in MODELS:
    for ds in DATASETS:
        # load all 3 prompt variants
        data_by_prompt = {}
        for pt in PROMPT_TYPES:
            p = load_paired(short, ds, pt, OUT_DIR)
            if p is not None:
                data_by_prompt[pt] = p

        available = list(data_by_prompt.keys())
        if len(available) < 2:
            print(f'SKIP {short}|{ds} — only {available} available')
            continue

        for cal_p, test_p in permutations(available, 2):
            cal_pairs  = data_by_prompt[cal_p]
            test_pairs = data_by_prompt[test_p]
            # align on shared IDs
            cal_ids  = {str(d['id']) for d, _ in cal_pairs}
            test_ids = {str(d['id']) for d, _ in test_pairs}
            shared   = cal_ids & test_ids
            cal_pairs  = [(d, r) for d, r in cal_pairs  if str(d['id']) in shared]
            test_pairs = [(d, r) for d, r in test_pairs if str(d['id']) in shared]
            # use first half as cal, second half as test (50/50)
            n_cal = len(cal_pairs) // 2
            res = run_cp_pair(cal_pairs[:n_cal], test_pairs[n_cal:])
            rows.append({
                'Model':    MODEL_LABELS.get(short, short),
                'Dataset':  DS_LABELS.get(ds, ds),
                'Cal→Test': f'{cal_p}→{test_p}',
                'Std CR%':  round(res['std']['cr'], 1),
                'Std SS':   round(res['std']['ss'], 2),
                'WCP CR%':  round(res['wcp']['cr'], 1),
                'WCP SS':   round(res['wcp']['ss'], 2),
                'ΔCR (pp)': round(res['wcp']['cr'] - res['std']['cr'], 1),
            })

df = pd.DataFrame(rows)
print(f'Cross-prompt pairs tested: {len(df)}')
print(f'Target coverage: {(1-ALPHA)*100:.0f}%\n')
pd.set_option('display.max_rows', 100)
df

In [ ]:
# ── Figure: coverage comparison across all cross-prompt pairs ─────────────────
if len(df) > 0:
    pairs = df['Cal→Test'].unique()
    x = np.arange(len(pairs))
    w = 0.35

    # Average across all models and datasets
    grp = df.groupby('Cal→Test')[['Std CR%', 'WCP CR%']].mean()
    grp = grp.reindex(pairs)

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(x - w/2, grp['Std CR%'], w, color='#4878CF', alpha=0.85, label='Standard CP')
    ax.bar(x + w/2, grp['WCP CR%'], w, color='#D65F5F', alpha=0.85, label='Weighted CP')
    ax.axhline(90, color='black', lw=1.5, ls='--', label='90% target')
    ax.set_xticks(x)
    ax.set_xticklabels(pairs, rotation=20, ha='right')
    ax.set_ylabel('Coverage Rate (%)')
    ax.set_ylim(50, 110)
    ax.set_title('Standard CP vs Weighted CP — avg coverage across all models × datasets\n(cal prompt → test prompt)', fontsize=11)
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/fig_wcp_cross_prompt.pdf', bbox_inches='tight')
    plt.show()
    print('Saved fig_wcp_cross_prompt.pdf')

    # ── Summary: how often does weighted CP recover coverage? ─────────────────
    below_std = (df['Std CR%'] < 90).sum()
    below_wcp = (df['WCP CR%'] < 90).sum()
    improved  = ((df['Std CR%'] < 90) & (df['WCP CR%'] >= 90)).sum()
    print(f'\nOut of {len(df)} cross-prompt evaluations:')
    print(f'  Standard CP below 90%: {below_std}')
    print(f'  Weighted CP below 90%: {below_wcp}')
    print(f'  Coverage recovered by weighted CP: {improved}')
    print(f'  Mean ΔCR (weighted − standard): {df["ΔCR (pp)"].mean():+.2f} pp')